# AnimalPoseTracker Demo

This notebook runs the Trimouse end-to-end workflow on a Kaggle GPU: training, independent validation on `val`, prediction on `test`, explicit ONNX export, and prediction with the exported ONNX model. Training defaults to 10 epochs; exporting is an explicit step and does not happen automatically after training.

Enable GPU and Internet in Kaggle Notebook Settings, then attach the Trimouse data under the Kaggle input dataset `AnimalDatasets`. The notebook searches only within that input dataset and reads files directly from `/kaggle/input` without copying them. Repository and project outputs are stored under `/kaggle/working/wux024`. Save a notebook version to retain the output files.

In [ ]:
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

import torch

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORKSPACE_ROOT = Path("/kaggle/working/wux024")
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)


def command_output(args):
    try:
        result = subprocess.run(args, capture_output=True, text=True, check=False)
        return result.stdout.strip() if result.returncode == 0 else result.stderr.strip()
    except FileNotFoundError:
        return "not available"


def read_meminfo_bytes(key):
    meminfo_path = Path("/proc/meminfo")
    if not meminfo_path.is_file():
        return None
    for line in meminfo_path.read_text(encoding="utf-8").splitlines():
        if line.startswith(f"{key}:"):
            return int(line.split()[1]) * 1024
    return None


cpu_model = platform.processor() or "unknown"
cpuinfo_path = Path("/proc/cpuinfo")
if cpuinfo_path.is_file():
    for line in cpuinfo_path.read_text(encoding="utf-8", errors="replace").splitlines():
        if line.lower().startswith(("model name", "hardware")):
            cpu_model = line.split(":", 1)[-1].strip()
            break

gpu_devices = []
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        gpu_devices.append({
            "index": index,
            "name": properties.name,
            "total_memory_bytes": properties.total_memory,
            "compute_capability": f"{properties.major}.{properties.minor}",
        })

runtime_report = {
    "python": sys.version,
    "platform": platform.platform(),
    "architecture": platform.machine(),
    "cpu_model": cpu_model,
    "logical_cpu_count": os.cpu_count(),
    "ram_total_bytes": read_meminfo_bytes("MemTotal"),
    "ram_available_bytes": read_meminfo_bytes("MemAvailable"),
    "kaggle_working_disk_total_bytes": shutil.disk_usage("/kaggle/working").total,
    "kaggle_working_disk_free_bytes": shutil.disk_usage("/kaggle/working").free,
    "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda,
    "cudnn_version": torch.backends.cudnn.version(),
    "cuda_available": torch.cuda.is_available(),
    "gpu_devices": gpu_devices,
    "nvidia_smi": command_output([
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total",
        "--format=csv,noheader",
    ]),
    "python_packages_before_install": {},
}

for package in (
    "numpy", "torchvision", "opencv-contrib-python-headless", "opencv-python-headless",
    "PyYAML", "scipy", "scikit-learn", "matplotlib", "onnx", "onnxsim",
    "onnxruntime", "onnxruntime-gpu", "pycocotools",
):
    try:
        runtime_report["python_packages_before_install"][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        runtime_report["python_packages_before_install"][package] = "not installed"

print("Kaggle runtime configuration:")
print(json.dumps(runtime_report, indent=2, ensure_ascii=False))
if not torch.cuda.is_available():
    raise RuntimeError("没有检测到 GPU。请在 Kaggle Notebook Settings 中启用 GPU 后重试。")

In [ ]:
if not KAGGLE_INPUT_ROOT.is_dir():
    raise FileNotFoundError("Kaggle input root does not exist: /kaggle/input")
input_items = sorted(path.name for path in KAGGLE_INPUT_ROOT.iterdir())
print("Attached Kaggle inputs:", input_items)
print("Workspace:", WORKSPACE_ROOT)
print("Input datasets are read-only; outputs will be written under /kaggle/working.")

In [ ]:
import json
from datetime import datetime
from pathlib import Path

REPO_URL = "https://github.com/wux024/AnimalPoseTracker.git"
BRANCH = "preview"
KAGGLE_COLLECTION_NAME = "AnimalDatasets"
# Set this explicitly if there are multiple Trimouse folders inside AnimalDatasets.
# Example: KAGGLE_COLLECTION_ROOT / "Trimouse"
DATA_ROOT_OVERRIDE = None

RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
REPO_DIR = WORKSPACE_ROOT / "AnimalPoseTracker"
PROJECTS_ROOT = WORKSPACE_ROOT

EPOCHS = 10
BATCH_SIZE = 4
IMAGE_SIZE = 640
NUM_WORKERS = 2
DEVICE = "0"
PREDICT_SPLIT = "test"

def normalise_name(value):
    return "".join(char for char in value.casefold() if char.isalnum())


collection_matches = [
    path for path in KAGGLE_INPUT_ROOT.iterdir()
    if path.is_dir() and normalise_name(KAGGLE_COLLECTION_NAME) in normalise_name(path.name)
]
if len(collection_matches) != 1:
    raise FileNotFoundError(
        f"Expected one Kaggle input matching {KAGGLE_COLLECTION_NAME!r}; "
        f"available inputs: {[path.name for path in KAGGLE_INPUT_ROOT.iterdir()]}"
    )
KAGGLE_COLLECTION_ROOT = collection_matches[0]

print("Run ID:", RUN_ID)
print("Branch:", BRANCH)
print("AnimalDatasets input:", KAGGLE_COLLECTION_ROOT)
print("Dataset override:", DATA_ROOT_OVERRIDE)
print("Repository:", REPO_DIR)
print("Project/output root:", PROJECTS_ROOT)

In [ ]:
import subprocess

WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
if REPO_DIR.exists():
    if not (REPO_DIR / ".git").exists():
        raise FileExistsError(f"Repository path exists but is not a Git checkout: {REPO_DIR}")
    repo_branch = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "branch", "--show-current"],
        text=True,
    ).strip()
    if repo_branch != BRANCH:
        raise RuntimeError(
            f"Expected branch {BRANCH!r} at {REPO_DIR}, found {repo_branch!r}. "
            "Check out the intended branch before continuing."
        )
    tracked_changes = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain", "--untracked-files=no"],
        text=True,
    ).strip()
    if tracked_changes:
        print("Existing checkout has tracked local changes; preserving it without updating.")
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "merge", "--ff-only", f"origin/{BRANCH}"],
            check=True,
        )
else:
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
repo_branch = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "branch", "--show-current"],
    text=True,
).strip()
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print("Repository branch:", repo_branch)
print("Repository commit:", commit)

In [ ]:
%cd {REPO_DIR}
%pip install -v --no-deps -e .
%pip install -q pycocotools onnx onnxsim scikit-learn

# Install CPU ONNX Runtime only when neither CPU nor GPU ONNX Runtime is already present.
if (
    runtime_report["python_packages_before_install"].get("onnxruntime") == "not installed"
    and runtime_report["python_packages_before_install"].get("onnxruntime-gpu") == "not installed"
):
    get_ipython().run_line_magic("pip", "install -q onnxruntime")

import importlib
import importlib.metadata

importlib.invalidate_caches()
import animalposetracker
print("Notebook kernel import:", animalposetracker.__file__)

# Verify the editable install in a fresh process, as used by project worker subprocesses.
%cd /tmp
!{sys.executable} -c "import animalposetracker; print('Worker import:', animalposetracker.__file__)"
%cd {REPO_DIR}

runtime_report["python_packages_after_install"] = {}
for package in (
    "animalposetracker", "pycocotools", "onnx", "onnxsim", "onnxruntime",
    "onnxruntime-gpu", "scikit-learn",
):
    try:
        runtime_report["python_packages_after_install"][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        runtime_report["python_packages_after_install"][package] = "not installed"
try:
    import onnxruntime as ort
    runtime_report["onnxruntime_providers"] = ort.get_available_providers()
except ImportError:
    runtime_report["onnxruntime_providers"] = []
print("Package versions after installation:")
print(json.dumps(runtime_report["python_packages_after_install"], indent=2, ensure_ascii=False))
print("ONNX Runtime providers:", runtime_report["onnxruntime_providers"])


In [ ]:
import yaml


def resolve_trimouse_root():
    if DATA_ROOT_OVERRIDE is not None:
        root = Path(DATA_ROOT_OVERRIDE).expanduser().resolve()
        if not (root / "images").is_dir():
            raise FileNotFoundError(f"No images/ directory under DATA_ROOT_OVERRIDE: {root}")
        return root

    image_dirs = [
        path for path in KAGGLE_COLLECTION_ROOT.rglob("images")
        if path.is_dir()
        and (path / "train").is_dir()
        and (path / "val").is_dir()
        and (path / "test").is_dir()
    ]
    roots = sorted({path.parent.resolve() for path in image_dirs})
    trimouse_roots = [
        root for root in roots
        if any("trimouse" in "".join(ch for ch in part.casefold() if ch.isalnum()) for part in root.parts)
    ]
    if len(trimouse_roots) == 1:
        return trimouse_roots[0]
    if len(roots) == 1:
        return roots[0]
    candidates = trimouse_roots or roots
    raise RuntimeError(
        "Could not select one Trimouse dataset automatically inside AnimalDatasets. "
        f"Candidates: {[str(path) for path in candidates]}. "
        "Set DATA_ROOT_OVERRIDE to the dataset folder containing images/."
    )


DATA_SOURCE_ROOT = resolve_trimouse_root()
DATA_YAML = yaml.safe_load(
    (REPO_DIR / "animalposetracker" / "cfg" / "datasets" / "trimouse.yaml")
    .read_text(encoding="utf-8")
)
DATA_YAML["path"] = str(DATA_SOURCE_ROOT)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
image_counts = {}
for split in ("train", "val", "test"):
    split_dir = DATA_SOURCE_ROOT / "images" / split
    if not split_dir.is_dir():
        raise FileNotFoundError(f"Missing split directory: {split_dir}")
    image_counts[split] = sum(
        1 for path in split_dir.rglob("*") if path.suffix.lower() in image_extensions
    )
    if image_counts[split] == 0:
        raise ValueError(f"No images found in {split_dir}")

ann_dir = DATA_SOURCE_ROOT / "annotations"
coco_candidates = [
    ann_dir / f"{split}.json" for split in ("train", "val", "test")
] + [
    ann_dir / f"person_keypoints_{split}2017.json"
    for split in ("train", "val", "test")
]
if not DATA_YAML.get("annotation_format"):
    DATA_YAML["annotation_format"] = (
        "coco" if any(path.is_file() for path in coco_candidates) else "yolo"
    )

print("Trimouse input (read-only):", DATA_SOURCE_ROOT)
print("Annotation format:", DATA_YAML["annotation_format"])
print("Image counts:", image_counts)
print("Keypoint shape:", DATA_YAML["kpt_shape"])
print("Classes:", DATA_YAML["names"])

In [ ]:
from animalposetracker.projector.animalposeproject import AnimalPoseTrackerProject

keypoint_count, keypoint_dims = DATA_YAML["kpt_shape"]
class_names = {
    int(class_id): str(name)
    for class_id, name in DATA_YAML["names"].items()
}
ordered_classes = [class_names[index] for index in sorted(class_names)]
skeleton = DATA_YAML.get("skeleton", [])
keypoint_names = DATA_YAML.get(
    "keypoint_names",
    [f"kpt_{index}" for index in range(keypoint_count)],
)

project = AnimalPoseTrackerProject(
    local_path=PROJECTS_ROOT,
    project_name="trimouse",
    worker="kaggle",
    model_type="AnimalRTPose",
    model_scale="N",
    keypoints=keypoint_count,
    visible=(keypoint_dims == 3),
    classes=len(class_names),
    keypoints_name=keypoint_names,
    skeleton=skeleton,
    kpt_oks_sigmas=DATA_YAML.get("kpt_oks_sigmas"),
    classes_name=ordered_classes,
    date=RUN_ID,
)
if project.project_path.exists():
    raise FileExistsError(f"Project path already exists: {project.project_path}")
project.create_new_project()
project.dataset_config.update(DATA_YAML)
project.dataset_config["path"] = str(DATA_SOURCE_ROOT)
project.dataset_config["names"] = class_names
project.project_config.update({
    "classes": len(class_names),
    "classes_name": ordered_classes,
    "keypoints": keypoint_count,
    "keypoints_name": keypoint_names,
    "visible": (keypoint_dims == 3),
    "skeleton": skeleton,
    "project_path": str(project.project_path),
})
project._sync_model_config_from_dataset()
project.other_config.update({
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "imgsz": IMAGE_SIZE,
    "workers": NUM_WORKERS,
    "device": DEVICE,
    "pretrained": True,
    "val": True,
    "plots": True,
    "project": "runs",
    "name": "train",
    "exist_ok": False,
})
project.save_configs("all")
runtime_report["project_path"] = str(project.project_path)
(project.project_path / "colab_environment.json").write_text(
    json.dumps(runtime_report, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
installed_packages = sorted(
    f"{distribution.metadata['Name']}=={distribution.version}"
    for distribution in importlib.metadata.distributions()
    if distribution.metadata.get("Name")
)
(project.project_path / "environment_packages.txt").write_text(
    "\n".join(installed_packages) + "\n",
    encoding="utf-8",
)
print("Project:", project.project_path)
print("Repository:", REPO_DIR)
print("Pretrained folder:", project.project_path / "pretrained")

In [ ]:
def run_stage(stage_name, start):
    result = start()
    process = project.process
    if process is None:
        raise RuntimeError(f"{stage_name}: no worker process was started")
    return_code = process.wait()
    print(f"{stage_name} exit code:", return_code)
    if return_code != 0:
        raise RuntimeError(f"{stage_name} failed; inspect worker output above")
    return result

run_stage("training", project.train)
run_name = str(project.other_config.get("name", "train"))
run_root = Path(project.other_config.get("project", "runs"))
if not run_root.is_absolute():
    run_root = project.project_path / run_root
train_dir = run_root / run_name
best_checkpoint = train_dir / "weights" / "best.pt"
last_checkpoint = train_dir / "weights" / "last.pt"
if not best_checkpoint.is_file() or not last_checkpoint.is_file():
    raise FileNotFoundError("Training did not produce both best.pt and last.pt")
print("Best checkpoint:", best_checkpoint)
print("Last checkpoint:", last_checkpoint)

In [ ]:
run_stage("independent val evaluation", project.evaluate)
metric_files = sorted(
    (project.project_path / "runs").glob("val*/metrics.json"),
    key=lambda path: path.stat().st_mtime,
)
if not metric_files:
    raise FileNotFoundError("No validation metrics.json was produced")
validation_metrics_path = metric_files[-1]
validation_metrics = json.loads(validation_metrics_path.read_text(encoding="utf-8"))
print("Validation metrics:", validation_metrics_path)
print(json.dumps(validation_metrics, indent=2, ensure_ascii=False))

In [ ]:
native_prediction_dir = run_stage(
    "test prediction from best.pt",
    lambda: project.predict(split=PREDICT_SPLIT),
)
native_predictions_path = native_prediction_dir / "predictions.json"
if not native_predictions_path.is_file():
    raise FileNotFoundError(f"Missing predictions: {native_predictions_path}")
native_predictions = json.loads(native_predictions_path.read_text(encoding="utf-8"))
print("Images predicted from .pt:", len(native_predictions))
print("Prediction folder:", native_prediction_dir)

In [ ]:
run_stage("explicit ONNX export", lambda: project.export(format="onnx"))
export_dirs = sorted(
    (project.project_path / "runs").glob("export*"),
    key=lambda path: path.stat().st_mtime,
)
if not export_dirs:
    raise FileNotFoundError("No export directory was produced")
onnx_model = export_dirs[-1] / "best.onnx"
if not onnx_model.is_file():
    raise FileNotFoundError(f"Missing ONNX model: {onnx_model}")
print("ONNX model:", onnx_model)

native_model_setting = project.other_config.get("model")
project.other_config["model"] = str(onnx_model)
project.save_configs("other")
try:
    onnx_prediction_dir = run_stage(
        "test prediction from ONNX",
        lambda: project.predict(split=PREDICT_SPLIT),
    )
    onnx_predictions_path = onnx_prediction_dir / "predictions.json"
    if not onnx_predictions_path.is_file():
        raise FileNotFoundError(f"Missing exported-model predictions: {onnx_predictions_path}")
    onnx_predictions = json.loads(onnx_predictions_path.read_text(encoding="utf-8"))
    print("Images predicted from ONNX:", len(onnx_predictions))
finally:
    project.other_config["model"] = native_model_setting
    project.save_configs("other")

In [ ]:
project_path = project.project_path

# Restore portable paths in the saved project configuration.
dataset_snapshot_path = project_path / "configs" / "dataset.yaml"
dataset_snapshot = yaml.safe_load(dataset_snapshot_path.read_text(encoding="utf-8"))
dataset_snapshot["path"] = str(DATA_SOURCE_ROOT)
dataset_snapshot_path.write_text(
    yaml.safe_dump(dataset_snapshot, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

other_snapshot_path = project_path / "configs" / "other.yaml"
other_snapshot = yaml.safe_load(other_snapshot_path.read_text(encoding="utf-8"))
other_snapshot["model"] = "configs/model.yaml"
other_snapshot["data"] = "configs/dataset.yaml"
pretrained_file = project_path / "pretrained" / "animalrtpose-n.pt"
def rewrite_pretrained_path(node):
    if isinstance(node, dict):
        for key, value in list(node.items()):
            if (
                key == "pretrained"
                and isinstance(value, str)
                and Path(value).name.lower() == pretrained_file.name
                and pretrained_file.is_file()
            ):
                node[key] = "pretrained/animalrtpose-n.pt"
            else:
                rewrite_pretrained_path(value)
    elif isinstance(node, list):
        for value in node:
            rewrite_pretrained_path(value)

rewrite_pretrained_path(other_snapshot)
other_snapshot_path.write_text(
    yaml.safe_dump(other_snapshot, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

# Save the full installed-package inventory alongside the hardware/runtime report.
runtime_report["project_path"] = str(project_path)
(project_path / "colab_environment.json").write_text(
    json.dumps(runtime_report, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
def relative_project_path(path):
    try:
        return str(Path(path).resolve().relative_to(project_path.resolve()))
    except ValueError:
        return str(path)

summary = {
    "repository": REPO_URL,
    "branch": BRANCH,
    "commit": commit,
    "model": "AnimalRTPose-N",
    "dataset_root": str(DATA_SOURCE_ROOT),
    "runtime": "Kaggle Notebook",
    "annotation_format": DATA_YAML["annotation_format"],
    "image_counts": image_counts,
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "validation_metrics": relative_project_path(validation_metrics_path),
    "native_prediction_dir": relative_project_path(native_prediction_dir),
    "onnx_model": relative_project_path(onnx_model),
    "onnx_prediction_dir": relative_project_path(onnx_prediction_dir),
    "saved_project": str(project_path),
}
(project_path / "colab_run_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print("Saved project and outputs under:", project_path)
print("Save a Kaggle Notebook version to retain /kaggle/working outputs.")
print("Environment report:", project_path / "colab_environment.json")
print("Installed package inventory:", project_path / "environment_packages.txt")
print(json.dumps(summary, indent=2, ensure_ascii=False))